In [11]:
### hivis image scraper

# takes the master sheet as input with usgs site number, base image url, and interval type
# also needs a time frame

# creates datasheets for hivis sites 

import requests
import pandas as pd
from functools import reduce, partial
from tqdm.notebook import tqdm

mastersheet = pd.read_excel('mastersheet_hivis.xlsx')

site_ids = (
    mastersheet['usgs_site_number']
    .astype(str)
    .apply(lambda s: s.zfill(8) if len(s) <= 8 else s.zfill(10))    
    .tolist()
)
base_image_url = mastersheet['hivis_base_url'].tolist()
intervals = mastersheet['time_step_hivis'].tolist()

interval_type = {
    5: [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55],
    10: [0, 10, 20, 30, 40, 50],
    15: [0, 15, 30, 45],
    30: [0, 30],
    60: [0],
}

startDT = '2026-01-01T00:01'
endDT = '2026-01-31T23:59'

def make_spreadsheet(site_id, startDT, endDT, out_file='usgs_data.xlsx'):
    # get json from USGS API
    base_url = "https://waterservices.usgs.gov/nwis/iv/"
    params = {
        "format": 'json',
        "sites": site_id,
        "startDT": startDT,
        "endDT": endDT,
        "siteStatus": 'all',
    }
    r = requests.get(base_url, params=params)
    r.raise_for_status()
    data = r.json()
    
    time_series = data['value']['timeSeries']
    
    dfs = []
    for series in time_series:
        site_name = series["sourceInfo"]["siteName"]
        site_code = series["sourceInfo"]["siteCode"][0]["value"]
        variable = series["variable"]["variableName"]
        unit = series["variable"]["unit"]["unitCode"]
    
        values = series["values"][0]["value"]
        if not values:
            continue
    
        df = pd.DataFrame(values)
        df["dateTime"] = (
            pd.to_datetime(df["dateTime"], errors='coerce', utc=True)
            .dt.tz_convert(None)
        )
        
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    
        # Rename value column to variable name
        col_name = f"{variable} ({unit})"
        df = df[["dateTime", "value"]].rename(columns={"value": col_name})
    
        dfs.append(df)

    # Combine into one dataframe
    if not dfs:
        print(f"No data for site {site_id}")
        return
    
    full_df = reduce(
        lambda left, right: pd.merge(left, right, on="dateTime", how="outer"),
        dfs
    )

    full_df["siteName"] = site_name
    full_df["siteCode"] = site_code

    full_df = full_df.sort_values("dateTime")

    # Save to Excel
    full_df.to_excel(out_file, index=False)

def append_hivis_url(dt, index):    
    if dt.minute not in interval_type[intervals[index]]:
        return
        
    for seconds in range(60):
        request_url = base_image_url[index].format(
            Y=dt.year, 
            M=f"{dt.month:02d}", 
            D=f"{dt.day:02d}", 
            h=f"{dt.hour:02d}", 
            m=f"{dt.minute:02d}", 
            s=f"{seconds:02d}"
        )
    
        try:
            # Send the HTTP GET request
            response = requests.get(request_url, stream=True)
            
            # Check response status
            #print(f"📡 Server responded with status code: {response.status_code}")
            
            if response.status_code == 200:
                #print(f"got 200 for {request_url}")
                return request_url
            
        except requests.exceptions.RequestException as e:
            print("⚠️ An error occurred:")
            print(e)
    
# loop thru all sites
for i in range(len(site_ids)):
    # create datasheet with usgs measurements
    make_spreadsheet(site_ids[i], startDT, endDT, out_file = 'usgs_' + site_ids[i] + '_datasheet_Jan26.xlsx')
    print(f"collected USGS data for site #{site_ids[i]}")
    
print("Collected USGS data for all sites.")

# find image url and append to spreadsheet
for i in range(len(site_ids)):
    try:
        df = pd.read_excel("usgs_" + site_ids[i] + "_datasheet_Jan26.xlsx")
    except:
        continue
    datetime_col = "dateTime"
    url_col = "image_url"
    df[datetime_col] = pd.to_datetime(df[datetime_col])

    tqdm.pandas()
    df[url_col] = df[datetime_col].progress_apply(
        partial(append_hivis_url, index=i)
    )
    df = df[df[url_col].notna()]

    df.to_excel("hivis_" + site_ids[i] + "_datasheet_Jan26.xlsx", index=False)
    print(f"collected {len(df)} image urls for site #{site_ids[i]}")
    
print("collected all image urls.")

collected USGS data for site #02146409
collected USGS data for site #07218525
collected USGS data for site #08062500
No data for site 05335450
collected USGS data for site #05335450
collected USGS data for site #11264500
collected USGS data for site #0137462010
collected USGS data for site #11510700
collected USGS data for site #01161280
collected USGS data for site #11176900
collected USGS data for site #01615000
No data for site 05404142
collected USGS data for site #05404142
collected USGS data for site #06463900
collected USGS data for site #01365100
collected USGS data for site #04208598
collected USGS data for site #01374901
collected USGS data for site #01185500
collected USGS data for site #01362487
collected USGS data for site #07350900
collected USGS data for site #06601200
collected USGS data for site #05429369
collected USGS data for site #0407991001
collected USGS data for site #06719505
collected USGS data for site #06709000
collected USGS data for site #04026511
collecte

  0%|          | 0/8927 [00:00<?, ?it/s]

⚠️ An error occurred:
HTTPSConnectionPool(host='usgs-nims-images.s3.amazonaws.com', port=443): Read timed out. (read timeout=None)
⚠️ An error occurred:
('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
⚠️ An error occurred:
('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
⚠️ An error occurred:
('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
⚠️ An error occurred:
('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
⚠️ An error occurred:
('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
collected 8917 image urls for site #02146409


  0%|          | 0/8927 [00:00<?, ?it/s]

⚠️ An error occurred:
HTTPSConnectionPool(host='usgs-nims-images.s3.amazonaws.com', port=443): Max retries exceeded with url: /720/NM_Sapello_River_above_Sapello/NM_Sapello_River_above_Sapello___2026-01-18T05-15-02Z.jpg (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001AF180C7AF0>: Failed to resolve 'usgs-nims-images.s3.amazonaws.com' ([Errno 11001] getaddrinfo failed)"))
⚠️ An error occurred:
HTTPSConnectionPool(host='usgs-nims-images.s3.amazonaws.com', port=443): Max retries exceeded with url: /720/NM_Sapello_River_above_Sapello/NM_Sapello_River_above_Sapello___2026-01-18T05-15-03Z.jpg (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001AF41F60340>: Failed to resolve 'usgs-nims-images.s3.amazonaws.com' ([Errno 11001] getaddrinfo failed)"))
⚠️ An error occurred:
HTTPSConnectionPool(host='usgs-nims-images.s3.amazonaws.com', port=443): Max retries exceeded with url: /720/NM_Sapello_River_above_Sapello/NM_Sapello_River

...

collected 1271 image urls for site #11264500


  0%|          | 0/2975 [00:00<?, ?it/s]

collected 335 image urls for site #0137462010


  0%|          | 0/8927 [00:00<?, ?it/s]

KeyboardInterrupt: 